# Importing positive and negative subsets of proteins from RNase experimental dataset, creating primary pairs with ORF2p and combined sequence FASTA files
*A)
 RNA-independent + I-DIRT significant (22)*

0. Start with the full experimental dataset - extracting only I-DIRT significant, RNA-independent proteins (pink and 3 blue)

In [1]:
import pandas as pd

# paths
IN_XLSX = "/mnt/storage/ana01/data/elife-30094-supp1-v6.xlsx"       
OUT_XLSX = "/mnt/storage/ana01/data/RNA_independent_subset.xlsx"
SHEET_NAME = "RNAse"  

# load 
df = pd.read_excel(IN_XLSX, sheet_name=SHEET_NAME)

# filter pink proteins + selected blue proteins 
keep_blue_genes = {"ORF2", "PABPC1", "PABPC4"}

df_pink = df[
    (df["Color"] == "pink") |
    ((df["Color"] == "blue") & (df["Genes"].isin(keep_blue_genes)))
].copy()

In [2]:
df = df_pink

# split multi-UniProt rows 
rows = []

for _, row in df.iterrows():
    uniprot_field = str(row["Uniprot"])

    # split on commas
    uniprot_ids = [u.strip() for u in uniprot_field.split(",") if u.strip()]

    for uid in uniprot_ids:
        new_row = row.copy()
        new_row["Protein ID"] = uid
        rows.append(new_row)

df_expanded = pd.DataFrame(rows)

# drop old column, keep clean contract 
df_expanded = df_expanded.drop(columns=["Uniprot"])

# optional sanity checks 
assert "Protein ID" in df_expanded.columns
assert df_expanded["Protein ID"].notna().all()

# write Excel for v2 scanner 
df_expanded.to_excel(OUT_XLSX, index=False)

print(f"Prepared {len(df_expanded)} UniProt IDs.")
print(df_expanded[["Genes", "Protein ID"]])


Prepared 22 UniProt IDs for Predictomes scan.
                      Genes Protein ID
4                    PABPC4     Q13310
5                    PABPC1     P11940
6                      ORF2     O00370
15                    FKBP4     Q02790
16                     HAX1     O00165
17                     PURB     Q96QR8
18                     IPO7     O95373
19                     TUBB     P07437
20                 HSP90AA1     P07900
21                 HSP90AB1     P08238
22                    PARP1     P09874
23           HSPA1A, HSPA1B     P0DMV8
23           HSPA1A, HSPA1B     P0DMV9
24                    HSPA8     P11142
25                     PCNA     P12004
26                   NAP1L1     P55209
27  UBB, UBC, RPS27A, UBA52     P0CG47
27  UBB, UBC, RPS27A, UBA52     P0CG48
27  UBB, UBC, RPS27A, UBA52     P62979
27  UBB, UBC, RPS27A, UBA52     P62987
28                   TUBB4B     P68371
29                     PURA     Q00577


1. Importing RNA-independent proteins






In [3]:
import pandas as pd

df = pd.read_excel("/mnt/storage/ana01/data/RNA_independent_subset.xlsx", header=0, engine="openpyxl")

print(f"Number of input proteins: {len(df)}")
df.head()



Number of input proteins: 22


,Genes,Protein,Color,P-value,Exp1_affinity,Exp2_affinity,Avg_affinity,Protein ID
0,PABPC4,Polyadenylate-binding protein 4,blue,NaN,-0.0655,-0.0547,-0.0601,Q13310
1,PABPC1,Polyadenylate-binding protein 1,blue,NaN,-0.0371,-0.0624,-0.0497,P11940
2,ORF2,NaN,blue,NaN,-0.1111,-0.0300,-0.0705,O00370
3,FKBP4,Peptidyl-prolyl cis-trans isomerase FKBP4,pink,NaN,0.0455,-0.1498,-0.0521,Q02790
4,HAX1,HCLS1-associated protein X-1,pink,NaN,-0.0952,-0.1327,-0.1139,O00165


2. Creating primary pairs of significant hits with the bait ('O00370', 'ORF2').



In [3]:
primary_pairs_df = pd.DataFrame()


primary_pairs_df['Protein A']= list(zip(df['Protein ID'], df['Genes']))
primary_pairs_df['Protein B']= [('O00370', 'ORF2')] * len(df)

primary_pairs_df.to_csv("/mnt/storage/ana01/results/RNase_22_primary_pairs.csv", index=False)

print(f"{len(primary_pairs_df)} primary pairs for input proteins saved to .../results/RNase_22_primary_pairs.csv")
primary_pairs_df.head()

22 primary pairs for input proteins saved to .../results/RNase_22_primary_pairs.csv


,Protein A,Protein B
0,"(Q13310, PABPC4)","(O00370, ORF2)"
1,"(P11940, PABPC1)","(O00370, ORF2)"
2,"(O00370, ORF2)","(O00370, ORF2)"
3,"(Q02790, FKBP4)","(O00370, ORF2)"
4,"(O00165, HAX1)","(O00370, ORF2)"


3.  Creating combined FASTA files for pairwise protein sequences
* INPUT: primary pairs CSV 
* fetching sequences from UniProt via UniProt ID
* cleaning gene names --> fasta headers --> file names (so they dont disrupt the pipeline downstream)
* combining 2 sequences and headers with clean names into one FASTA file 
* OUTPUT: FASTA files written to /mnt/storage/ana01/results/fasta_files_RNase_22


In [4]:
import requests
from textwrap import wrap
import os
import pandas as pd
import ast

#Import CSV of primary protein pairs
df = pd.read_csv("/mnt/storage/ana01/results/RNase_22_primary_pairs.csv")

#Create output directory for FASTA files
fasta_dir = "/mnt/storage/ana01/results/fasta_files_RNase_22"
os.makedirs(fasta_dir, exist_ok=True)


# Function that fetches individual protein fasta from UniProt
def fetch_uniprot_fasta(acc): 
     url = f"https://rest.uniprot.org/uniprotkb/{acc}.fasta"
     r = requests.get(url, timeout = 30)
     r.raise_for_status()
     return r.text.strip()

# Loop over each row in the dataframe and create FASTA files for each protein pair
tracker = 0 
for i in range(df.shape[0]):
    protein_A = ast.literal_eval(df.iloc[i,0])
    protein_B = ast.literal_eval(df.iloc[i,1])   

    if protein_A[0] != protein_B[0]:
       
        #Gene Name : UniProt ID dictionary 
        ACCESSIONS = { protein_A[1]: protein_A[0], protein_B[1]:protein_B[0] }
    
        # Clean gene names (e.g., MAGOH/B → MAGOHB)
        clean_names = {}
        for name, uid in ACCESSIONS.items():
            clean_name = name.replace("/", "")
            clean_names[clean_name] = uid

        # Fetch FASTA sequences and reformat
        records = []
        for name, ID in clean_names.items():
            raw_fasta = fetch_uniprot_fasta(ID)

            # Reformat 
            lines = raw_fasta.splitlines() 
            seq = "".join(line.strip() for line in lines[1:] if line and not line.startswith(">")) 

            # Use the clean name in the header
            new_header = f">{name}" 
            seq_wrapped = "\n".join(wrap(seq, 60)) 
            records.append(f"{new_header}\n{seq_wrapped}")

        combined_fasta = "\n".join(records) 

        # Write to disk
        keys = list(clean_names.keys())
        fasta_path = os.path.join(fasta_dir, f"{keys[0]}_{keys[1]}_multimer.fasta")

        with open(fasta_path, "w") as f:
            f.write(combined_fasta) 
        
        tracker += 1
    
    else:  #for homodimers ORF2-ORF2 case
        #Gene Name : UniProt ID dictionary --> collapse to single entry
        ACCESSION = { protein_A[1]: protein_A[0]}
    
        # Clean gene names (e.g., MAGOH/B → MAGOHB)
        clean_names = {}
        for name, uid in ACCESSION.items():
            clean_name = name.replace("/", "")
            clean_names[clean_name] = uid

        # Fetch FASTA sequences and reformat
        records = []
        for name, ID in clean_names.items():
            raw_fasta = fetch_uniprot_fasta(ID)

            # Reformat 
            lines = raw_fasta.splitlines() 
            seq = "".join(line.strip() for line in lines[1:] if line and not line.startswith(">")) 

            # Use the clean name in the header
            new_header = f">{name}" 
            seq_wrapped = "\n".join(wrap(seq, 60)) 
            records.append(f"{new_header}\n{seq_wrapped}")

            raw_fasta = fetch_uniprot_fasta(ID)

            # Reformat 
            lines = raw_fasta.splitlines() 
            seq = "".join(line.strip() for line in lines[1:] if line and not line.startswith(">")) 

            # Use the clean name in the header
            new_header = f">{name}" 
            seq_wrapped = "\n".join(wrap(seq, 60)) 
            records.append(f"{new_header}\n{seq_wrapped}")

        combined_fasta = "\n".join(records) 

        # Write to disk
        keys = list(clean_names.keys())
        fasta_path = os.path.join(fasta_dir, f"{keys[0]}_{keys[0]}_multimer.fasta")

        with open(fasta_path, "w") as f:
            f.write(combined_fasta) 
        
        tracker += 1


print(f"{tracker} FASTA files written to: {fasta_dir}\n")


22 FASTA files written to: /mnt/storage/ana01/results/fasta_files_RNase_22



*B) RNA-dependent I-DIRT significant (blue - 3 used above) (4) and insignificant (black) (8) in batch*

1. Extract the RNA-dependent subset and clean different UniProt IDs into different rows

In [5]:
# paths
IN_XLSX = "/mnt/storage/ana01/data/elife-30094-supp1-v6.xlsx"       
OUT_XLSX = "/mnt/storage/ana01/data/RNA_dependent_subset.xlsx"
SHEET_NAME = "RNAse"  


df = pd.read_excel(IN_XLSX, sheet_name=SHEET_NAME)

remove_blue_genes = {"ORF2", "PABPC1", "PABPC4"}

df_blue_black = df[
    ((df["Color"] == "blue") & ~df["Genes"].isin(remove_blue_genes)) |
    (df["Color"] == "black")
].copy()

In [6]:
df = df_blue_black

# split multi-UniProt rows 
rows = []

rows = []

for _, row in df.iterrows():
    uniprot_ids = [u.strip() for u in str(row["Uniprot"]).split(",") if u.strip()]
    genes = [g.strip() for g in str(row["Genes"]).split(",") if g.strip()]

    # if counts match, split in parallel; otherwise fall back safely
    if len(uniprot_ids) == len(genes):
        pairs = zip(uniprot_ids, genes)
    else:
        pairs = [(uid, row["Genes"]) for uid in uniprot_ids]

    for uid, gene in pairs:
        new_row = row.copy()
        new_row["Protein ID"] = uid
        new_row["Genes"] = gene
        rows.append(new_row)


df_expanded = pd.DataFrame(rows)

# drop old column, keep clean contract 
df_expanded = df_expanded.drop(columns=["Uniprot"])

#clean ORF1 gene name L1RE (ORF1) to ORF1 - so that FASTA file naming is uninterupted
df_expanded["Genes"] = df_expanded["Genes"].replace({"L1RE (ORF1)": "ORF1"})

#  checks 
assert "Protein ID" in df_expanded.columns
assert df_expanded["Protein ID"].notna().all()

# write Excel for v2 scanner 
df_expanded.to_excel(OUT_XLSX, index=False)

print(f"Prepared {len(df_expanded)} UniProt IDs.")



Prepared 13 UniProt IDs.


The number is 13, as one row carries 2 proteins.

I adapted the uniprot ID splitting function to also split gene names in the corresponding row assuming matching gene name 1 - uniprot 1 and so on. For this case, the assumption was confirmed in uniprot manually. 

1. Primary pairs


In [7]:
df = pd.read_excel("/mnt/storage/ana01/data/RNA_dependent_subset.xlsx", header=0, engine="openpyxl")

primary_pairs_df = pd.DataFrame()


primary_pairs_df['Protein A']= list(zip(df['Protein ID'], df['Genes']))
primary_pairs_df['Protein B']= [('O00370', 'ORF2')] * len(df)

primary_pairs_df.to_csv("/mnt/storage/ana01/results/RNase_13_primary_pairs.csv", index=False)

print(f"{len(primary_pairs_df)} primary pairs for input proteins saved to .../results/RNase_22_primary_pairs.csv")


13 primary pairs for input proteins saved to .../results/RNase_22_primary_pairs.csv


2. FASTA fetching

In [8]:
import requests
from textwrap import wrap
import os
import pandas as pd
import ast

#Import CSV of primary protein pairs
df = pd.read_csv("/mnt/storage/ana01/results/RNase_13_primary_pairs.csv")

#Create output directory for FASTA files
fasta_dir = "/mnt/storage/ana01/results/fasta_files_RNase_13"
os.makedirs(fasta_dir, exist_ok=True)


# Function that fetches individual protein fasta from UniProt
def fetch_uniprot_fasta(acc): 
     url = f"https://rest.uniprot.org/uniprotkb/{acc}.fasta"
     r = requests.get(url, timeout = 30)
     r.raise_for_status()
     return r.text.strip()

# Loop over each row in the dataframe and create FASTA files for each protein pair
tracker = 0 
for i in range(df.shape[0]):
    protein_A = ast.literal_eval(df.iloc[i,0])
    protein_B = ast.literal_eval(df.iloc[i,1])   

    if protein_A[0] != protein_B[0]:
       
        #Gene Name : UniProt ID dictionary 
        ACCESSIONS = { protein_A[1]: protein_A[0], protein_B[1]:protein_B[0] }
    
        # Clean gene names (e.g., MAGOH/B → MAGOHB)
        clean_names = {}
        for name, uid in ACCESSIONS.items():
            clean_name = name.replace("/", "")
            clean_names[clean_name] = uid

        # Fetch FASTA sequences and reformat
        records = []
        for name, ID in clean_names.items():
            raw_fasta = fetch_uniprot_fasta(ID)

            # Reformat 
            lines = raw_fasta.splitlines() 
            seq = "".join(line.strip() for line in lines[1:] if line and not line.startswith(">")) 

            # Use the clean name in the header
            new_header = f">{name}" 
            seq_wrapped = "\n".join(wrap(seq, 60)) 
            records.append(f"{new_header}\n{seq_wrapped}")

        combined_fasta = "\n".join(records) 

        # Write to disk
        keys = list(clean_names.keys())
        fasta_path = os.path.join(fasta_dir, f"{keys[0]}_{keys[1]}_multimer.fasta")

        with open(fasta_path, "w") as f:
            f.write(combined_fasta) 
        
        tracker += 1
    
    else:  #for homodimers ORF2-ORF2 case
        #Gene Name : UniProt ID dictionary --> collapse to single entry
        ACCESSION = { protein_A[1]: protein_A[0]}
    
        # Clean gene names (e.g., MAGOH/B → MAGOHB)
        clean_names = {}
        for name, uid in ACCESSION.items():
            clean_name = name.replace("/", "")
            clean_names[clean_name] = uid

        # Fetch FASTA sequences and reformat
        records = []
        for name, ID in clean_names.items():
            raw_fasta = fetch_uniprot_fasta(ID)

            # Reformat 
            lines = raw_fasta.splitlines() 
            seq = "".join(line.strip() for line in lines[1:] if line and not line.startswith(">")) 

            # Use the clean name in the header
            new_header = f">{name}" 
            seq_wrapped = "\n".join(wrap(seq, 60)) 
            records.append(f"{new_header}\n{seq_wrapped}")

            raw_fasta = fetch_uniprot_fasta(ID)

            # Reformat 
            lines = raw_fasta.splitlines() 
            seq = "".join(line.strip() for line in lines[1:] if line and not line.startswith(">")) 

            # Use the clean name in the header
            new_header = f">{name}" 
            seq_wrapped = "\n".join(wrap(seq, 60)) 
            records.append(f"{new_header}\n{seq_wrapped}")

        combined_fasta = "\n".join(records) 

        # Write to disk
        keys = list(clean_names.keys())
        fasta_path = os.path.join(fasta_dir, f"{keys[0]}_{keys[0]}_multimer.fasta")

        with open(fasta_path, "w") as f:
            f.write(combined_fasta) 
        
        tracker += 1


print(f"{tracker} FASTA files written to: {fasta_dir}\n")


13 FASTA files written to: /mnt/storage/ana01/results/fasta_files_RNase_13

